# Tamil Nadu Multi-City Climate — Interactive Folium Map
**Dataset:** `data/processed/tamil_nadu_climate.csv`  
**Cities:** Chennai · Coimbatore · Madurai · Tiruchirappalli · Vellore · Tirunelveli · Thanjavur · Cuddalore  
**Parameters:** GHI · T_ambient · Wind Speed · Precipitation · Cloud Cover · Relative Humidity  
**Resolution:** Hourly | Year: 2024

## Cell 1 — Install & Import Dependencies

In [ ]:
# Install required libraries
%pip install folium branca plotly pandas numpy -q

import pandas as pd
import numpy as np
import folium
from folium import plugins
from folium.plugins import HeatMap, MiniMap, Fullscreen, MousePosition
import branca.colormap as cm
import branca
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from IPython.display import IFrame, display
import warnings
warnings.filterwarnings('ignore')

print('[OK] All libraries loaded successfully!')

Note: you may need to restart the kernel to use updated packages.
[OK] All libraries loaded successfully!


## Cell 2 — Load Data & Map Cities

In [ ]:
# ─── CONFIG ───────────────────────────────────────────────────────────────────
CSV_PATH = 'data/era5_climate_tamilnadu_2024.csv'

CITY_MAP = {
    (13.0,  80.25): 'Chennai',        (13.0,  80.0):  'Chennai',
    (13.25, 80.25): 'Chennai',        (12.75, 80.0):  'Chennai',
    (11.0,  77.0):  'Coimbatore',     (11.0,  76.75): 'Coimbatore',
    (10.75, 76.75): 'Coimbatore',     (11.25, 76.75): 'Coimbatore',
    (9.75,  78.0):  'Madurai',        (9.75,  77.75): 'Madurai',
    (10.0,  78.0):  'Madurai',        (10.75, 78.5):  'Tiruchirappalli',
    (10.5,  78.25): 'Tiruchirappalli',(12.5,  79.25): 'Vellore',
    (8.5,   76.75): 'Tirunelveli',    (10.0,  79.0):  'Thanjavur',
    (11.5,  79.5):  'Cuddalore',
}

# City display colours (for markers & line plots)
CITY_COLORS = {
    'Chennai':         '#e74c3c',
    'Coimbatore':      '#f39c12',
    'Madurai':         '#2ecc71',
    'Tiruchirappalli': '#3498db',
    'Vellore':         '#9b59b6',
    'Tirunelveli':     '#1abc9c',
    'Thanjavur':       '#e67e22',
    'Cuddalore':       '#e91e8c',
}

CITY_ABBR = {
    'Chennai': 'CHN', 'Coimbatore': 'CBE', 'Madurai': 'MDU',
    'Tiruchirappalli': 'TRZ', 'Vellore': 'VEL', 'Tirunelveli': 'TEN',
    'Thanjavur': 'TJR', 'Cuddalore': 'CUD',
}
# ──────────────────────────────────────────────────────────────────────────────

# Load CSV
df = pd.read_csv(CSV_PATH, parse_dates=['timestamp'])
df.columns = df.columns.str.strip()

# Round lat/lon to 2 decimal places to match CITY_MAP keys
df['lat_r'] = df['latitude'].round(2)
df['lon_r'] = df['longitude'].round(2)

# Map (lat, lon) → city name; unmapped points → 'Unknown'
df['city'] = df.apply(
    lambda r: CITY_MAP.get((r['lat_r'], r['lon_r']), 'Unknown'), axis=1
)

# Drop unmapped points
df_mapped = df[df['city'] != 'Unknown'].copy()

# Time components
df_mapped['hour']   = df_mapped['timestamp'].dt.hour
df_mapped['month']  = df_mapped['timestamp'].dt.month
df_mapped['DOY']    = df_mapped['timestamp'].dt.dayofyear
df_mapped['date']   = df_mapped['timestamp'].dt.date

# Season mapping
def get_season(m):
    if m in [12, 1, 2]:     return 'Winter'
    elif m in [3, 4, 5]:    return 'Pre-Monsoon'
    elif m in [6, 7, 8, 9]: return 'Monsoon'
    else:                   return 'Post-Monsoon'

df_mapped['season_label'] = df_mapped['month'].apply(get_season)

print(f'[OK] Loaded : {len(df):,} rows  |  Mapped: {len(df_mapped):,} rows')
print(f'   Cities : {sorted(df_mapped["city"].unique())}')
print(f'   Date   : {df_mapped["timestamp"].min()} → {df_mapped["timestamp"].max()}')
display(df_mapped.groupby("city")[["GHI_Wm2","T_ambient_C","Wind_speed_ms","Precip_mm","RH_percent"]].mean().round(2))

[OK] Loaded : 3,434,544 rows  |  Mapped: 149,328 rows
   Cities : ['Chennai', 'Coimbatore', 'Cuddalore', 'Madurai', 'Thanjavur', 'Tiruchirappalli', 'Tirunelveli', 'Vellore']
   Date   : 2024-01-01 00:00:00 → 2024-12-31 23:00:00


,GHI_Wm2,T_ambient_C,Wind_speed_ms,Precip_mm,RH_percent
city,,,,,
Chennai,32.44,29.01,3.33,0.15,76.73
Coimbatore,32.80,25.45,2.20,0.18,74.96
Cuddalore,32.72,29.14,2.65,0.20,76.27
Madurai,33.61,29.13,1.69,0.14,69.15
Thanjavur,33.84,29.37,2.95,0.15,77.01
Tiruchirappalli,33.19,29.06,2.53,0.13,69.83
Tirunelveli,33.47,27.64,3.94,0.26,81.65
Vellore,32.44,28.39,2.34,0.17,74.20


## Cell 3 — Pre-compute Aggregates & Helpers

In [ ]:
# ── City-level annual stats ──────────────────────────────────────────────────
city_stats = df_mapped.groupby('city').agg(
    lat_c   =('latitude',            'mean'),
    lon_c   =('longitude',           'mean'),
    GHI_mean=('GHI_Wm2',            'mean'),
    GHI_max =('GHI_Wm2',            'max'),
    T_mean  =('T_ambient_C',         'mean'),
    T_max   =('T_ambient_C',         'max'),
    T_min   =('T_ambient_C',         'min'),
    Wind_mean=('Wind_speed_ms',       'mean'),
    Precip_sum=('Precip_mm',         'sum'),
    Cloud_mean=('Cloud_cover_fraction','mean'),
    RH_mean =('RH_percent',          'mean'),
    RH_max  =('RH_percent',          'max'),
).reset_index()

# ── Grid-point annual stats ──────────────────────────────────────────────────
grid_stats = df_mapped.groupby(['latitude','longitude','city']).agg(
    GHI_mean=('GHI_Wm2','mean'),
    T_mean  =('T_ambient_C','mean'),
    count=('timestamp','count')
).reset_index()

# ── Monthly stats per city ───────────────────────────────────────────────────
df_mapped['month'] = df_mapped['timestamp'].dt.month
monthly = df_mapped.groupby(['city','month']).agg(
    GHI_mean  =('GHI_Wm2',             'mean'),
    GHI_max   =('GHI_Wm2',             'max'),
    T_mean    =('T_ambient_C',          'mean'),
    T_max     =('T_ambient_C',          'max'),
    Wind_mean =('Wind_speed_ms',        'mean'),
    Wind_max  =('Wind_speed_ms',        'max'),
    Precip_sum=('Precip_mm',           'sum'),
    RH_mean   =('RH_percent',          'mean'),
    Cloud_mean=('Cloud_cover_fraction','mean'),
).reset_index()

# ── Hourly pattern (average by hour) ──────────────────────────────────────────
hourly_pattern = df_mapped.groupby(['city','hour']).agg(
    GHI_mean=('GHI_Wm2','mean'),
    T_mean=('T_ambient_C','mean'),
    RH_mean=('RH_percent','mean'),
).reset_index()

# ── Seasonal stats ────────────────────────────────────────────────────────────
seasonal = df_mapped.groupby(['city','season_label']).agg(
    GHI_mean=('GHI_Wm2','mean'),
    T_mean=('T_ambient_C','mean'),
    Precip_sum=('Precip_mm','sum'),
    RH_mean=('RH_percent','mean'),
    count=('timestamp','count')
).reset_index()

# ── Normaliser helper ────────────────────────────────────────────────────────
def norm_series(s, lo=None, hi=None):
    lo = lo if lo is not None else s.min()
    hi = hi if hi is not None else s.max()
    return ((s - lo) / (hi - lo + 1e-9)).clip(0, 1)

# Global min/max for normalisation
G = {
    'GHI':   (df_mapped['GHI_Wm2'].min(),              df_mapped['GHI_Wm2'].max()),
    'T':     (df_mapped['T_ambient_C'].min(),           df_mapped['T_ambient_C'].max()),
    'RH':    (df_mapped['RH_percent'].min(),            df_mapped['RH_percent'].max()),
    'Wind':  (df_mapped['Wind_speed_ms'].min(),         df_mapped['Wind_speed_ms'].max()),
    'Precip':(0,                                        df_mapped['Precip_mm'].quantile(0.99)),
    'Cloud': (df_mapped['Cloud_cover_fraction'].min(),  df_mapped['Cloud_cover_fraction'].max()),
}

# ── Popup builder ────────────────────────────────────────────────────────────
def city_popup(row):
    color = CITY_COLORS.get(row['city'], '#aaaaaa')
    html = f"""
    <div style='font-family:Arial,sans-serif;font-size:13px;width:250px;
                background:#1a1f2e;color:#ecf0f1;padding:14px;border-radius:10px;
                border:1px solid {color}44;'>
      <div style='font-size:16px;font-weight:bold;color:{color};margin-bottom:8px;'>
        [CITY] {row['city']}
      </div>
      <hr style='border-color:#2c3e50;margin:6px 0;'/>
      <table width='100%' style='border-collapse:collapse;'>
        <tr><td style='padding:3px 0;color:#95a5a6;'>[GHI] mean</td>
            <td align='right'><b style='color:#f39c12;'>{row['GHI_mean']:.1f} W/m²</b></td></tr>
        <tr><td style='padding:3px 0;color:#95a5a6;'>[GHI] max</td>
            <td align='right'><b style='color:#e74c3c;'>{row['GHI_max']:.1f} W/m²</b></td></tr>
        <tr><td style='padding:3px 0;color:#95a5a6;'>[TEMP] T mean</td>
            <td align='right'><b style='color:#e67e22;'>{row['T_mean']:.1f} °C</b></td></tr>
        <tr><td style='padding:3px 0;color:#95a5a6;'>[TEMP] T range</td>
            <td align='right'>{row['T_min']:.1f}–{row['T_max']:.1f} °C</td></tr>
        <tr><td style='padding:3px 0;color:#95a5a6;'>[WIND] mean</td>
            <td align='right'><b style='color:#1abc9c;'>{row['Wind_mean']:.2f} m/s</b></td></tr>
        <tr><td style='padding:3px 0;color:#95a5a6;'>[RH] mean</td>
            <td align='right'><b style='color:#2ecc71;'>{row['RH_mean']:.1f} %</b></td></tr>
        <tr><td style='padding:3px 0;color:#95a5a6;'>[PRECIP] total</td>
            <td align='right'><b style='color:#3498db;'>{row['Precip_sum']:.1f} mm</b></td></tr>
        <tr><td style='padding:3px 0;color:#95a5a6;'>[CLOUD]</td>
            <td align='right'>{row['Cloud_mean']:.3f}</td></tr>
      </table>
    </div>"""
    return folium.Popup(folium.IFrame(html, width=275, height=260), max_width=290)

def grid_popup(row):
    html = f"""
    <div style='font-family:Arial,sans-serif;font-size:12px;width:200px;
                background:#1a1f2e;color:#ecf0f1;padding:10px;border-radius:8px;'>
      <b style='color:#3498db;'>Grid Point</b><br/>
      <hr style='border-color:#2c3e50;margin:5px 0;'/>
      [CITY] {row['city']}<br/>
      [LOC] Lat: {row['latitude']:.2f} | Lon: {row['longitude']:.2f}<br/>
      [GHI] mean: <b style='color:#f39c12;'>{row['GHI_mean']:.1f} W/m²</b><br/>
      [TEMP] mean: <b style='color:#e67e22;'>{row['T_mean']:.1f} °C</b><br/>
      [DATA] {int(row['count'])} records
    </div>"""
    return folium.Popup(folium.IFrame(html, width=220, height=160), max_width=235)

print('[OK] Aggregates and helpers ready.')
print(f'   City centroids: {len(city_stats)} | Grid points: {len(grid_stats)}')
print(f'   Monthly records: {len(monthly)} | Hourly patterns: {len(hourly_pattern)}')

[OK] Aggregates and helpers ready.
   City centroids: 8 | Grid points: 17
   Monthly records: 96 | Hourly patterns: 192


## Cell 4 — Build the Multi-Layer Folium Map

In [ ]:
# ─── Base map: centred on Tamil Nadu ─────────────────────────────────────────
m = folium.Map(
    location=[11.0, 78.5],
    zoom_start=7,
    tiles=None,
    prefer_canvas=True
)

# ── Tile layers ───────────────────────────────────────────────────────────────
folium.TileLayer('CartoDB dark_matter', name='Dark (default)', control=False).add_to(m)
folium.TileLayer('CartoDB positron',    name='Light').add_to(m)
folium.TileLayer('OpenStreetMap',       name='OpenStreetMap').add_to(m)
folium.TileLayer(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    attr='Esri', name='Satellite'
).add_to(m)

# ── Plugins ───────────────────────────────────────────────────────────────────
Fullscreen(position='topleft', title='Fullscreen').add_to(m)
MiniMap(toggle_display=True, tile_layer='CartoDB positron', zoom_level_offset=-5).add_to(m)
MousePosition(position='bottomleft', numDigits=4, prefix='Lat/Lon:').add_to(m)

# ─── Subsample helper for large heatmaps ─────────────────────────────────────
MAX_HEAT = 15000  # cap points per heatmap layer for performance

def build_heat(sub_df, val_col, lo, hi, frac=1.0):
    """Return [[lat, lon, intensity], ...] normalised 0-1."""
    s = sub_df.sample(min(MAX_HEAT, len(sub_df)), random_state=42) if frac < 1.0 or len(sub_df) > MAX_HEAT else sub_df
    vals = norm_series(s[val_col], lo, hi).values
    return [[r.latitude, r.longitude, v] for r, v in zip(s.itertuples(), vals)]

# ─── Layer 1: GHI Heatmap (daytime only) ─────────────────────────────────────
df_day = df_mapped[df_mapped['GHI_Wm2'] > 0]
HeatMap(
    build_heat(df_day, 'GHI_Wm2', *G['GHI']),
    name='[GHI] Heatmap', show=True,
    min_opacity=0.35, radius=22, blur=18,
    gradient={0.0:'#0d1b2a', 0.25:'#1a6b9a', 0.55:'#f39c12',
              0.8:'#e74c3c', 1.0:'#ffffff'}
).add_to(m)

# ─── Layer 2: Temperature Heatmap ────────────────────────────────────────────
HeatMap(
    build_heat(df_mapped, 'T_ambient_C', *G['T']),
    name='[TEMP] Temperature Heatmap', show=False,
    min_opacity=0.4, radius=22, blur=18,
    gradient={0.0:'#2980b9', 0.3:'#82e0aa', 0.65:'#f9e79f',
              0.85:'#e67e22', 1.0:'#c0392b'}
).add_to(m)

# ─── Layer 3: Humidity Heatmap ────────────────────────────────────────────────
HeatMap(
    build_heat(df_mapped, 'RH_percent', *G['RH']),
    name='[HUMIDITY] Humidity Heatmap', show=False,
    min_opacity=0.4, radius=22, blur=18,
    gradient={0.0:'#fef9e7', 0.35:'#a9cce3',
              0.65:'#1a9850', 1.0:'#145a32'}
).add_to(m)

# ─── Layer 4: Wind Speed Heatmap ─────────────────────────────────────────────
HeatMap(
    build_heat(df_mapped, 'Wind_speed_ms', *G['Wind']),
    name='[WIND] Wind Speed Heatmap', show=False,
    min_opacity=0.4, radius=22, blur=18,
    gradient={0.0:'#e8f8f5', 0.3:'#76d7c4',
              0.65:'#1abc9c', 1.0:'#0e6655'}
).add_to(m)

# ─── Layer 5: Precipitation Heatmap (rainy hours only) ───────────────────────
df_rain = df_mapped[df_mapped['Precip_mm'] > 0]
if len(df_rain) > 0:
    HeatMap(
        build_heat(df_rain, 'Precip_mm', *G['Precip']),
        name='[PRECIP] Precipitation Heatmap', show=False,
        min_opacity=0.45, radius=24, blur=20,
        gradient={0.0:'#d6eaf8', 0.35:'#5dade2',
                  0.7:'#1a5276', 1.0:'#0a2942'}
    ).add_to(m)

# ─── Layer 6: Cloud Cover Heatmap ────────────────────────────────────────────
HeatMap(
    build_heat(df_mapped, 'Cloud_cover_fraction', *G['Cloud']),
    name='[CLOUD] Cloud Cover Heatmap', show=False,
    min_opacity=0.35, radius=22, blur=18,
    gradient={0.0:'#212f3c', 0.3:'#566573',
              0.65:'#aab7b8', 1.0:'#f0f3f4'}
).add_to(m)

# ─── Layer 7: City Summary Markers ───────────────────────────────────────────
fg_city = folium.FeatureGroup(name='[CITY] City Summary Markers', show=True)
for _, row in city_stats.iterrows():
    color = CITY_COLORS.get(row['city'], '#aaaaaa')
    abbr  = CITY_ABBR.get(row['city'], row['city'][:3].upper())
    icon_html = f"""
    <div style='background:{color};color:#fff;border-radius:50%;
                width:42px;height:42px;display:flex;align-items:center;
                justify-content:center;font-size:10px;font-weight:bold;
                font-family:Arial;border:2.5px solid rgba(255,255,255,0.8);
                box-shadow:0 2px 6px rgba(0,0,0,0.5);'>{abbr}</div>"""
    folium.Marker(
        location=[row['lat_c'], row['lon_c']],
        icon=folium.DivIcon(html=icon_html, icon_size=(42, 42), icon_anchor=(21, 21)),
        popup=city_popup(row),
        tooltip=f"{row['city']} | GHI={row['GHI_mean']:.0f} W/m² | T={row['T_mean']:.1f}°C"
    ).add_to(fg_city)
fg_city.add_to(m)

# ─── Layer 8: Grid Point Markers ─────────────────────────────────────────────
fg_grid = folium.FeatureGroup(name='[GRID] Grid Point Markers', show=False)
ghi_cmap = cm.LinearColormap(
    ['#0d1b2a','#2e86c1','#f39c12','#ffffff'],
    vmin=grid_stats['GHI_mean'].min(),
    vmax=grid_stats['GHI_mean'].max(),
    caption='Mean GHI (W/m²) — grid points'
)
for _, row in grid_stats.iterrows():
    fc = ghi_cmap(row['GHI_mean'])
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=7, color='#ffffff', weight=0.8,
        fill=True, fill_color=fc, fill_opacity=0.85,
        tooltip=f"[LOC] {row['city']} | ({row['latitude']:.2f}, {row['longitude']:.2f}) | GHI={row['GHI_mean']:.1f} W/m²",
        popup=grid_popup(row)
    ).add_to(fg_grid)
fg_grid.add_to(m)

# ─── Layer 9: Peak GHI Events (top 1%) ───────────────────────────────────────
fg_peak  = folium.FeatureGroup(name='[PEAK] Peak GHI Events (top 1%)', show=False)
thresh   = df_mapped['GHI_Wm2'].quantile(0.99)
df_peak  = df_mapped[df_mapped['GHI_Wm2'] >= thresh]
# Sample max 800 for performance
df_peak_s = df_peak.sample(min(800, len(df_peak)), random_state=7)
for r in df_peak_s.itertuples():
    city_col = CITY_COLORS.get(r.city, '#e74c3c')
    peak_html = f"""
    <div style='font-family:Arial;font-size:12px;width:210px;
                background:#1a1f2e;color:#ecf0f1;padding:10px;border-radius:8px;
                border:1px solid #e74c3c44;'>
      <b style='color:#f39c12;'>[PEAK] Peak Solar Event</b><br/>
      <hr style='border-color:#2c3e50;margin:5px 0;'/>
      [CITY] City: <b style='color:{city_col};'>{r.city}</b><br/>
      [TIME] {r.timestamp}<br/>
      [GHI] GHI: <b style='color:#f39c12;'>{r.GHI_Wm2:.1f} W/m²</b><br/>
      [TEMP] T_amb: {r.T_ambient_C:.1f} °C<br/>
      [LOC] ({r.latitude:.2f}, {r.longitude:.2f})
    </div>"""
    folium.CircleMarker(
        location=[r.latitude, r.longitude],
        radius=6, color='#e74c3c', weight=1.5,
        fill=True, fill_color='#f39c12', fill_opacity=0.9,
        tooltip=f"[PEAK] GHI={r.GHI_Wm2:.0f} W/m² | {r.city} | {r.timestamp}",
        popup=folium.Popup(folium.IFrame(peak_html, width=230, height=155), max_width=245)
    ).add_to(fg_peak)
fg_peak.add_to(m)

# ─── Colour bar legend ────────────────────────────────────────────────────────
ghi_cmap.add_to(m)

# ─── Layer control ────────────────────────────────────────────────────────────
folium.LayerControl(position='topright', collapsed=False).add_to(m)

print('[OK] Map built!')
print(f'   Total mapped rows  : {len(df_mapped):,}')
print(f'   Daytime rows (GHI>0): {len(df_day):,}')
print(f'   Rainy hours        : {len(df_rain):,}')
print(f'   Peak GHI threshold : {thresh:.1f} W/m²  ({len(df_peak):,} events)')

# Display the main map
m

[OK] Map built!
   Total mapped rows  : 149,328
   Daytime rows (GHI>0): 43,179
   Rainy hours        : 74,588
   Peak GHI threshold : 245.8 W/m²  (1,496 events)


## Cell 5 — Inject Fixed Stats Panel (HTML Overlay)

## Cell 5A — Temperature Heatmap by City

In [ ]:
# ─── MAP 5A: Temperature Heatmap ────────────────────────────────────────────
m_temp = folium.Map(
    location=[11.0, 78.5],
    zoom_start=7,
    tiles='CartoDB dark_matter'
)

df_day_temp = df_mapped
temp_cmap = cm.LinearColormap(
    ['#2980b9', '#82e0aa', '#f9e79f', '#e67e22', '#c0392b'],
    vmin=G['T'][0], vmax=G['T'][1],
    caption='Temperature (°C)'
)

# Sample for heatmap
np.random.seed(42)
df_temp_sample = df_day_temp.sample(min(10000, len(df_day_temp)), random_state=42)
temp_vals = df_temp_sample['T_ambient_C'].values
temp_norm = (temp_vals - G['T'][0]) / (G['T'][1] - G['T'][0] + 1e-9)
heat_temp = [[r.latitude, r.longitude, float(v)] for r, v in zip(df_temp_sample.itertuples(), temp_norm)]

HeatMap(heat_temp, radius=20, blur=15, min_opacity=0.3,
        gradient={0.0: '#2980b9', 0.3: '#82e0aa', 0.65: '#f9e79f', 0.85: '#e67e22', 1.0: '#c0392b'}
).add_to(m_temp)

# City markers
for _, row in city_stats.iterrows():
    color = CITY_COLORS.get(row['city'], '#aaa')
    folium.CircleMarker(
        location=[row['lat_c'], row['lon_c']],
        radius=8, color=color, fill=True, fill_color=color, fill_opacity=0.8,
        popup=city_popup(row),
        tooltip=f"{row['city']} | T: {row['T_mean']:.1f}°C"
    ).add_to(m_temp)

temp_cmap.add_to(m_temp)
folium.LayerControl().add_to(m_temp)
print('[OK] Temperature heatmap created')
m_temp

## Cell 5B — Humidity & Wind Rose Map

In [ ]:
# ─── MAP 5B: Humidity Heatmap + Wind Vectors ────────────────────────────────
m_wind = folium.Map(
    location=[11.0, 78.5],
    zoom_start=7,
    tiles='CartoDB positron'
)

# Humidity heatmap
df_wind_sample = df_mapped.sample(min(8000, len(df_mapped)), random_state=43)
rh_vals = df_wind_sample['RH_percent'].values
rh_norm = (rh_vals - G['RH'][0]) / (G['RH'][1] - G['RH'][0] + 1e-9)
heat_rh = [[r.latitude, r.longitude, float(v)] for r, v in zip(df_wind_sample.itertuples(), rh_norm)]

HeatMap(heat_rh, radius=18, blur=15, min_opacity=0.3,
        gradient={0.0: '#fef9e7', 0.3: '#aed6f1', 0.65: '#2eab4f', 1.0: '#0d5016'}
).add_to(m_wind)

# Wind direction vectors (sample by city)
for city in df_mapped['city'].unique():
    city_data = df_mapped[df_mapped['city'] == city].sample(min(100, len(df_mapped[df_mapped['city'] == city])))
    lat_c = city_data['latitude'].mean()
    lon_c = city_data['longitude'].mean()
    
    wind_avg = city_data['Wind_speed_ms'].mean()
    
    # Create a circle marker showing wind conditions
    folium.CircleMarker(
        location=[lat_c, lon_c],
        radius=max(5, wind_avg * 2),
        color=CITY_COLORS.get(city, '#aaa'),
        fill=True, fill_color=CITY_COLORS.get(city, '#aaa'), fill_opacity=0.6,
        popup=city_popup(city_stats[city_stats['city'] == city].iloc[0]),
        tooltip=f"{city} | Wind: {wind_avg:.2f} m/s | RH: {city_data['RH_percent'].mean():.1f}%"
    ).add_to(m_wind)

rh_cmap = cm.LinearColormap(['#fef9e7', '#aed6f1', '#2eab4f', '#0d5016'],
                             vmin=G['RH'][0], vmax=G['RH'][1],
                             caption='Relative Humidity (%)')
rh_cmap.add_to(m_wind)
folium.LayerControl().add_to(m_wind)
print('[OK] Humidity & wind map created')
m_wind

## Cell 5C — Precipitation & Cloud Cover Map

In [ ]:
# ─── MAP 5C: Precipitation & Cloud Cover ────────────────────────────────────
m_precip = folium.Map(
    location=[11.0, 78.5],
    zoom_start=7,
    tiles='CartoDB dark_matter'
)

# Cloud cover heatmap
df_cloud_sample = df_mapped.sample(min(8000, len(df_mapped)), random_state=44)
cloud_vals = df_cloud_sample['Cloud_cover_fraction'].values
cloud_norm = (cloud_vals - G['Cloud'][0]) / (G['Cloud'][1] - G['Cloud'][0] + 1e-9)
heat_cloud = [[r.latitude, r.longitude, float(v)] for r, v in zip(df_cloud_sample.itertuples(), cloud_norm)]

HeatMap(heat_cloud, radius=20, blur=15, min_opacity=0.3,
        gradient={0.0: '#212f3c', 0.3: '#7f8c8d', 0.65: '#bdc3c7', 1.0: '#ecf0f1'}
).add_to(m_precip)

# Monthly precipitation summary by city
for _, row in city_stats.iterrows():
    precip_pct = (row['Precip_sum'] / df_mapped['Precip_mm'].sum()) * 100 if df_mapped['Precip_mm'].sum() > 0 else 0
    
    # Adjust radius based on precipitation
    radius = max(6, row['Precip_sum'] / 10)
    
    folium.CircleMarker(
        location=[row['lat_c'], row['lon_c']],
        radius=radius, color='#3498db', fill=True, fill_color='#5dade2', fill_opacity=0.7,
        popup=folium.Popup(f"<b>{row['city']}</b><br>[PRECIP] {row['Precip_sum']:.1f} mm ({precip_pct:.1f}%)<br>[CLOUD] {row['Cloud_mean']:.2f}", max_width=200),
        tooltip=f"{row['city']} | Precip: {row['Precip_sum']:.1f} mm"
    ).add_to(m_precip)

cloud_cmap = cm.LinearColormap(['#212f3c', '#7f8c8d', '#bdc3c7', '#ecf0f1'],
                               vmin=G['Cloud'][0], vmax=G['Cloud'][1],
                               caption='Cloud Cover Fraction (0–1)')
cloud_cmap.add_to(m_precip)
folium.LayerControl().add_to(m_precip)
print('[OK] Precipitation & cloud map created')
m_precip

## Cell 6 — Plotly Interactive Charts: City Comparisons

In [ ]:
# ─── PLOTLY 1: City Annual Stats Comparison (Radar Charts) ──────────────────
# Normalize stats for radar chart (0-100 scale)
city_radar = city_stats.copy()
city_radar['GHI_norm'] = (city_radar['GHI_mean'] - city_radar['GHI_mean'].min()) / (city_radar['GHI_mean'].max() - city_radar['GHI_mean'].min() + 1e-9) * 100
city_radar['T_norm'] = (city_radar['T_mean'] - city_radar['T_mean'].min()) / (city_radar['T_mean'].max() - city_radar['T_mean'].min() + 1e-9) * 100
city_radar['Wind_norm'] = (city_radar['Wind_mean'] - city_radar['Wind_mean'].min()) / (city_radar['Wind_mean'].max() - city_radar['Wind_mean'].min() + 1e-9) * 100
city_radar['Precip_norm'] = (city_radar['Precip_sum'] - city_radar['Precip_sum'].min()) / (city_radar['Precip_sum'].max() - city_radar['Precip_sum'].min() + 1e-9) * 100
city_radar['RH_norm'] = (city_radar['RH_mean'] - city_radar['RH_mean'].min()) / (city_radar['RH_mean'].max() - city_radar['RH_mean'].min() + 1e-9) * 100

fig_radar = go.Figure()

for _, row in city_radar.iterrows():
    fig_radar.add_trace(go.Scatterpolar(
        r=[row['GHI_norm'], row['T_norm'], row['Wind_norm'], row['RH_norm'], row['Precip_norm']],
        theta=['[GHI] Solar', '[TEMP] Heat', '[WIND]', '[RH] Humidity', '[PRECIP]'],
        fill='toself',
        name=row['city'],
        line_color=CITY_COLORS.get(row['city'], '#aaa')
    ))

fig_radar.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 100])),
    title='[CITY] Climate Profiles — Normalized Comparison',
    font=dict(family='Arial, sans-serif', size=11, color='#aaa'),
    paper_bgcolor='#0f1419', plot_bgcolor='#1a1f2e',
    showlegend=True, hovermode='closest',
    height=600
)
fig_radar.show()

# ─── PLOTLY 2: Monthly GHI by City (Box Plot) ────────────────────────────────
fig_ghi = px.box(
    df_mapped, x='month', y='GHI_Wm2', color='city',
    title='[GHI] Monthly Distribution by City',
    labels={'month': '[MONTH]', 'GHI_Wm2': 'GHI (W/m²)', 'city': '[CITY]'},
    color_discrete_map=CITY_COLORS,
    height=500
)
fig_ghi.update_layout(paper_bgcolor='#0f1419', plot_bgcolor='#1a1f2e',
                      font=dict(color='#aaa'), hovermode='closest')
fig_ghi.show()

# ─── PLOTLY 3: Temperature Trends (Line Chart) ───────────────────────────────
daily_temp = df_mapped.groupby(['date', 'city']).agg(
    T_mean=('T_ambient_C', 'mean'),
    T_max=('T_ambient_C', 'max'),
    T_min=('T_ambient_C', 'min')
).reset_index()

fig_temp = px.line(
    daily_temp, x='date', y='T_mean', color='city',
    title='[TEMP] Daily Average Temperature Trends',
    labels={'date': '[DATE]', 'T_mean': 'Avg Temp (°C)', 'city': '[CITY]'},
    color_discrete_map=CITY_COLORS,
    height=500
)
fig_temp.update_layout(paper_bgcolor='#0f1419', plot_bgcolor='#1a1f2e',
                       font=dict(color='#aaa'), hovermode='x unified')
fig_temp.show()

print('[OK] Plotly charts rendered')

## Cell 7 — Plotly: Hourly & Seasonal Patterns

In [ ]:
# ─── PLOTLY 4: Hourly GHI Pattern (Area Chart) ───────────────────────────────
fig_hourly = px.area(
    hourly_pattern, x='hour', y='GHI_mean', color='city',
    title='[GHI] Mean Hourly Pattern by City',
    labels={'hour': '[HOUR] (0–23)', 'GHI_mean': 'Mean GHI (W/m²)', 'city': '[CITY]'},
    color_discrete_map=CITY_COLORS,
    height=500
)
fig_hourly.update_layout(paper_bgcolor='#0f1419', plot_bgcolor='#1a1f2e',
                         font=dict(color='#aaa'), hovermode='x unified')
fig_hourly.show()

# ─── PLOTLY 5: Seasonal Comparison (Grouped Bar) ──────────────────────────────
fig_seasonal = px.bar(
    seasonal, x='city', y='GHI_mean', color='season_label',
    title='[GHI] Seasonal Mean by City',
    labels={'GHI_mean': 'Mean GHI (W/m²)', 'city': '[CITY]', 'season_label': '[SEASON]'},
    barmode='group', height=500,
    color_discrete_map={'Winter': '#1565C0', 'Pre-Monsoon': '#F57C00', 
                       'Monsoon': '#2E7D32', 'Post-Monsoon': '#6A1B9A'}
)
fig_seasonal.update_layout(paper_bgcolor='#0f1419', plot_bgcolor='#1a1f2e',
                           font=dict(color='#aaa'), hovermode='closest')
fig_seasonal.show()

# ─── PLOTLY 6: Precipitation Heatmap (Monthly by City) ───────────────────────
pivoted_precip = monthly.pivot(index='month', columns='city', values='Precip_sum')
fig_precip_hm = go.Figure(data=go.Heatmap(
    z=pivoted_precip.values,
    x=pivoted_precip.columns,
    y=['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'],
    colorscale='Blues',
    hovertemplate='<b>%{x}</b><br>[MONTH] %{y}<br>[PRECIP] %{z:.1f} mm<extra></extra>'
))
fig_precip_hm.update_layout(
    title='[PRECIP] Monthly Accumulation by City (mm)',
    xaxis_title='[CITY]', yaxis_title='[MONTH]',
    height=500, paper_bgcolor='#0f1419', plot_bgcolor='#1a1f2e',
    font=dict(color='#aaa')
)
fig_precip_hm.show()

# ─── PLOTLY 7: Wind Speed Variation (Violin Plot) ──────────────────────────────
fig_wind = px.violin(
    df_mapped, x='city', y='Wind_speed_ms',
    title='[WIND] Speed Distribution by City',
    labels={'Wind_speed_ms': 'Wind Speed (m/s)', 'city': '[CITY]'},
    color='city', color_discrete_map=CITY_COLORS,
    height=500, box=True, points=False
)
fig_wind.update_layout(paper_bgcolor='#0f1419', plot_bgcolor='#1a1f2e',
                       font=dict(color='#aaa'), hovermode='closest')
fig_wind.show()

print('[OK] Hourly & seasonal charts rendered')

## Cell 8 — Summary Statistics & Export

In [ ]:
# ─── SUMMARY STATISTICS TABLE ────────────────────────────────────────────────
print('=' * 80)
print('[SUMMARY] Tamil Nadu Climate Statistics — 2024 Annual Report')
print('=' * 80)

summary_table = city_stats[['city','GHI_mean','GHI_max','T_mean','T_min','T_max',
                              'Wind_mean','RH_mean','Precip_sum','Cloud_mean']].copy()
summary_table.columns = ['City', 'GHI Avg', 'GHI Max', 'T Avg', 'T Min', 'T Max',
                         'Wind Avg', 'RH Avg', 'Precip Tot', 'Cloud Avg']

print('\n[ANNUAL] City-Level Summary:')
print(summary_table.to_string(index=False))

print('\n[MONTHLY] Top 3 GHI Months Overall:')
monthly_ghi_top = monthly.groupby('month').agg(GHI_avg=('GHI_mean','mean')).sort_values('GHI_avg', ascending=False).head(3)
for idx, (m, row) in enumerate(monthly_ghi_top.iterrows(), 1):
    print(f'  {idx}. Month {int(m)}: {row["GHI_avg"]:.1f} W/m²')

print('\n[SEASON] Seasonal Precipitation:')
seasonal_precip = seasonal.groupby('season_label')['Precip_sum'].sum().sort_values(ascending=False)
for season, precip in seasonal_precip.items():
    print(f'  {season}: {precip:.1f} mm')

# Export city stats to CSV
city_export_path = 'tamil_nadu_city_stats.csv'
summary_table.to_csv(city_export_path, index=False)
print(f'\n[EXPORT] City stats saved → {city_export_path}')

# Export monthly data to CSV
monthly_export_path = 'tamil_nadu_monthly_data.csv'
monthly_export = monthly[['city','month','GHI_mean','T_mean','Wind_mean','RH_mean','Precip_sum']]
monthly_export.to_csv(monthly_export_path, index=False)
print(f'[EXPORT] Monthly data saved → {monthly_export_path}')

# Display city comparison table
print('\n[TABLE] City Rankings:')
rankings = city_stats.copy()
rankings['GHI_rank'] = rankings['GHI_mean'].rank(ascending=False)
rankings['T_rank'] = rankings['T_mean'].rank(ascending=False)
rankings['Precip_rank'] = rankings['Precip_sum'].rank(ascending=False)
rankings['Wind_rank'] = rankings['Wind_mean'].rank(ascending=False)

rank_display = rankings[['city','GHI_rank','T_rank','Precip_rank','Wind_rank']]
rank_display.columns = ['City', '[GHI] Rank', '[TEMP] Rank', '[PRECIP] Rank', '[WIND] Rank']
print(rank_display.to_string(index=False))

print('\n' + '=' * 80)

In [ ]:
# Compute Tamil Nadu-wide annual summary
tn_ghi_mean   = df_mapped['GHI_Wm2'].mean()
tn_ghi_max    = df_mapped['GHI_Wm2'].max()
tn_t_mean     = df_mapped['T_ambient_C'].mean()
tn_wind_mean  = df_mapped['Wind_speed_ms'].mean()
tn_precip_sum = df_mapped['Precip_mm'].sum()
tn_rh_mean    = df_mapped['RH_percent'].mean()
tn_cloud_mean = df_mapped['Cloud_cover_fraction'].mean()
n_cities      = df_mapped['city'].nunique()
n_gridpts     = df_mapped[['latitude','longitude']].drop_duplicates().shape[0]

stats_html = f"""
<div style="
  position: fixed; bottom: 50px; left: 12px; z-index: 9999;
  background: rgba(15,19,30,0.93); color: #ecf0f1;
  padding: 14px 18px; border-radius: 12px;
  font-family: 'Courier New', monospace; font-size: 12px;
  border: 1px solid rgba(243,156,18,0.35);
  min-width: 215px; pointer-events: none;">
  <div style="color:#f39c12;font-size:13px;font-weight:bold;margin-bottom:9px;letter-spacing:.5px;">
    [TN] Tamil Nadu — 2024 Summary
  </div>
  <table style="border-collapse:collapse;width:100%;">
    <tr><td style="padding:2px 8px 2px 0;color:#7f8c8d;">Cities</td>
        <td style="text-align:right;">{n_cities} | {n_gridpts} grid pts</td></tr>
    <tr><td colspan='2'><hr style='border-color:#2c3e50;margin:5px 0;'/></td></tr>
    <tr><td style="padding:2px 8px 2px 0;color:#7f8c8d;">[GHI] mean</td>
        <td style="text-align:right;color:#f39c12;">{tn_ghi_mean:.1f} W/m²</td></tr>
    <tr><td style="padding:2px 8px 2px 0;color:#7f8c8d;">[GHI] max</td>
        <td style="text-align:right;color:#e74c3c;">{tn_ghi_max:.1f} W/m²</td></tr>
    <tr><td style="padding:2px 8px 2px 0;color:#7f8c8d;">[TEMP] T_ambient</td>
        <td style="text-align:right;color:#e67e22;">{tn_t_mean:.1f} °C</td></tr>
    <tr><td style="padding:2px 8px 2px 0;color:#7f8c8d;">[WIND]</td>
        <td style="text-align:right;color:#1abc9c;">{tn_wind_mean:.2f} m/s</td></tr>
    <tr><td style="padding:2px 8px 2px 0;color:#7f8c8d;">[RH] mean</td>
        <td style="text-align:right;color:#2ecc71;">{tn_rh_mean:.1f} %</td></tr>
    <tr><td style="padding:2px 8px 2px 0;color:#7f8c8d;">[PRECIP] (sum)</td>
        <td style="text-align:right;color:#3498db;">{tn_precip_sum:.0f} mm</td></tr>
    <tr><td style="padding:2px 8px 2px 0;color:#7f8c8d;">[CLOUD] mean</td>
        <td style="text-align:right;">{tn_cloud_mean:.3f}</td></tr>
  </table>
</div>
"""

m.get_root().html.add_child(folium.Element(stats_html))
print('[OK] Stats panel injected.')

# Display the map with injected stats panel
m

[OK] Stats panel injected.


## Cell 6 — Save HTML & Display Inline

In [ ]:
OUTPUT_PATH = 'tamil_nadu_climate_map.html'
m.save(OUTPUT_PATH)
print(f'[OK] Map saved -> {OUTPUT_PATH}')
IFrame(OUTPUT_PATH, width='100%', height=650)

[OK] Map saved -> tamil_nadu_climate_map.html


## Cell 7 — Download the HTML Map

In [ ]:
try:
    from google.colab import files
    files.download(OUTPUT_PATH)
    print('[OK] Download triggered!')
except ImportError:
    print(f'[INFO] Not in Colab — file is at: {OUTPUT_PATH}')

## Cell 8 — Bonus: Monthly Parameter Charts (3×2, Dark Theme, Per City)

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.lines as mlines

MONTH_LABELS = ['Jan','Feb','Mar','Apr','May','Jun',
                'Jul','Aug','Sep','Oct','Nov','Dec']
CITIES = sorted(df_mapped['city'].unique())

params = [
    ('GHI_mean',   'GHI Mean (W/m²)',      '#f39c12'),
    ('T_mean',     'T_ambient Mean (°C)',   '#e74c3c'),
    ('Wind_mean',  'Wind Speed Mean (m/s)', '#1abc9c'),
    ('Precip_sum', 'Precip Total (mm)',     '#3498db'),
    ('RH_mean',    'Relative Humidity (%)', '#2ecc71'),
    ('Cloud_mean', 'Cloud Cover Mean',      '#95a5a6'),
]

fig, axes = plt.subplots(3, 2, figsize=(16, 14), facecolor='#12131a')
fig.suptitle('Tamil Nadu 2024 — Monthly Climate Parameters by City',
             color='#f39c12', fontsize=15, fontweight='bold', y=1.01)

for ax, (col, label, _) in zip(axes.flat, params):
    ax.set_facecolor('#1a1f2e')
    for city in CITIES:
        sub = monthly[monthly['city'] == city].sort_values('month')
        if sub.empty:
            continue
        color = CITY_COLORS.get(city, '#aaaaaa')
        ax.plot(sub['month'], sub[col],
                color=color, linewidth=2.0, marker='o',
                markersize=4, alpha=0.9, label=city)
        ax.fill_between(sub['month'], sub[col],
                        alpha=0.06, color=color)

    ax.set_title(label, color='#ecf0f1', fontsize=11, pad=6)
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(MONTH_LABELS, fontsize=8, color='#95a5a6')
    ax.tick_params(axis='y', colors='#95a5a6', labelsize=8)
    ax.spines[:].set_color('#2c3e50')
    ax.grid(axis='y', color='#2c3e50', linewidth=0.5, linestyle='--', alpha=0.7)
    ax.grid(axis='x', color='#2c3e50', linewidth=0.3, linestyle=':', alpha=0.5)

# Shared legend
handles = [mlines.Line2D([], [], color=CITY_COLORS.get(c,'#aaa'),
            linewidth=2, marker='o', markersize=5, label=c) for c in CITIES]
fig.legend(handles=handles, loc='lower center', ncol=len(CITIES),
           frameon=True, facecolor='#1a1f2e', edgecolor='#2c3e50',
           labelcolor='#ecf0f1', fontsize=10, bbox_to_anchor=(0.5, -0.02))

plt.tight_layout()
CHART_PATH = 'tamil_nadu_monthly_params.png'
plt.savefig(CHART_PATH, dpi=150, bbox_inches='tight', facecolor='#12131a')
plt.show()
print(f'[OK] Chart saved -> {CHART_PATH}')

# Download chart too
try:
    from google.colab import files
    files.download(CHART_PATH)
except ImportError:
    pass